# 02 — Worked solutions: binary classification

The rows are synthetic; the scores below describe only this generated sample. The worked pipeline selects model regularization by training-fold cross-validation, chooses its threshold on validation data under the stated toy costs, and evaluates the frozen choices on the final test set once.

## Setup and Exercise 1

The setup creates distinct training, validation, and test partitions. The following cell fits a pipeline and selects its regularization using only training rows. Validation metrics at 0.5 are a reference; the final test set is not inspected yet.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_data_file(filename):
    for directory in (Path.cwd(), *Path.cwd().parents):
        data_path = directory / "data" / filename
        if data_path.is_file():
            return data_path
    raise FileNotFoundError(f"Could not find data/{filename}")


customer_data = pd.read_csv(find_data_file("churn.csv"))
feature_data = customer_data.drop(columns="churn")
churn_response = customer_data["churn"].astype(int)

(
    development_features,
    final_test_features,
    development_response,
    final_test_response,
) = train_test_split(
    feature_data,
    churn_response,
    test_size=0.20,
    random_state=19,
    stratify=churn_response,
)
(
    training_features,
    validation_features,
    training_response,
    validation_response,
) = train_test_split(
    development_features,
    development_response,
    test_size=0.25,
    random_state=20,
    stratify=development_response,
)
numeric_columns = ["tenure_months", "monthly_charge", "support_calls"]
categorical_columns = ["contract"]
preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric_columns,
        ),
        (
            "category",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_columns,
        ),
    ]
)
print("Training class proportions:")
print(training_response.value_counts(normalize=True).sort_index().round(3))
print(f"Training rows: {len(training_features)}")
print(f"Validation rows: {len(validation_features)}")
print(f"Final test rows: {len(final_test_features)}")

from sklearn.model_selection import GridSearchCV, StratifiedKFold

classification_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessing),
        ("model", LogisticRegression(max_iter=2000)),
    ]
)
regularization_search = GridSearchCV(
    estimator=classification_pipeline,
    param_grid={"model__C": [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]},
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=19),
    refit=True,
)
regularization_search.fit(training_features, training_response)
best_classifier = regularization_search.best_estimator_
validation_probabilities = best_classifier.predict_proba(validation_features)[:, 1]
validation_predictions = (validation_probabilities >= 0.5).astype(int)

majority_baseline = DummyClassifier(strategy="most_frequent")
majority_baseline.fit(training_features, training_response)
baseline_predictions = majority_baseline.predict(validation_features)
print("Selected C:", regularization_search.best_params_["model__C"])
print("Training-fold ROC-AUC:", round(regularization_search.best_score_, 3))
print("Validation baseline accuracy:", round(accuracy_score(validation_response, baseline_predictions), 3))
print("Validation metrics at 0.5:", {
    "accuracy": round(accuracy_score(validation_response, validation_predictions), 3),
    "precision": round(precision_score(validation_response, validation_predictions, zero_division=0), 3),
    "recall": round(recall_score(validation_response, validation_predictions, zero_division=0), 3),
    "F1": round(f1_score(validation_response, validation_predictions, zero_division=0), 3),
    "ROC-AUC": round(roc_auc_score(validation_response, validation_probabilities), 3),
})
print("Validation confusion matrix:\n", confusion_matrix(validation_response, validation_predictions))

If the non-churn class is more common, always predicting it can yield high accuracy while recall for churn is zero. This is why accuracy must be interpreted with the class distribution, confusion matrix, and the intended action.

## Exercise 2 — Cost rule and validation table

Predict churn at probability $p$ when its expected false-positive cost $C_{FP}(1-p)$ is no greater than the false-negative cost $C_{FN}p$. Rearranging gives $p\ge C_{FP}/(C_{FP}+C_{FN})$. For costs 1 and 5, the theoretical cutoff is $1/6\approx0.167$, assuming calibrated probabilities and constant known costs.

In [ ]:
false_positive_cost = 1.0
false_negative_cost = 5.0
threshold_candidates = [0.10, 0.17, 0.25, 0.35, 0.50, 0.65, 0.80]
validation_cost_rows = []

for threshold in threshold_candidates:
    validation_decisions = (validation_probabilities >= threshold).astype(int)
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        validation_response, validation_decisions, labels=[0, 1]
    ).ravel()
    validation_cost_rows.append(
        {
            "threshold": threshold,
            "false_positives": false_positive,
            "false_negatives": false_negative,
            "estimated_cost": (false_positive_cost * false_positive
                               + false_negative_cost * false_negative),
        }
    )

validation_cost_table = pd.DataFrame(validation_cost_rows).sort_values(
    ["estimated_cost", "threshold"]
)
validation_cost_table

The lowest cost here is selected from the validation partition. The final test set is used once below after fixing both model and threshold. This split helps prevent optimistic threshold selection; the synthetic costs still do not define any real business policy.

## Exercise 3 — Metric meaning

Precision is the fraction of flagged cases that are positive; recall is the fraction of all positives found. High recall may matter when missing a case is especially harmful and follow-up is cheap. High precision may matter when alerts trigger costly or intrusive actions. ROC-AUC measures ranking across cutoffs but contains no error costs, capacity constraint, or acceptable-risk preference.

## Exercise 4 — Final test

Regularization selection occurs inside cross-validation on training rows. The validation partition chooses the threshold under the assumed cost ratio. The final test partition now gives one assessment of those frozen choices.

In [ ]:
selected_threshold = float(validation_cost_table.iloc[0]["threshold"])
final_test_probabilities = best_classifier.predict_proba(final_test_features)[:, 1]
final_test_predictions = (final_test_probabilities >= selected_threshold).astype(int)

print(f"Threshold selected on validation: {selected_threshold:.2f}")
print("One final held-out evaluation:")
print({
    "accuracy": round(accuracy_score(final_test_response, final_test_predictions), 3),
    "precision": round(precision_score(final_test_response, final_test_predictions, zero_division=0), 3),
    "recall": round(recall_score(final_test_response, final_test_predictions, zero_division=0), 3),
    "F1": round(f1_score(final_test_response, final_test_predictions, zero_division=0), 3),
    "ROC-AUC": round(roc_auc_score(final_test_response, final_test_probabilities), 3),
})
print("Final test confusion matrix:\n", confusion_matrix(final_test_response, final_test_predictions))

## Exercise 5 — Limits

Support calls may reflect an omitted product problem that also drives churn; reducing calls would not necessarily change churn. Counts may be measured inconsistently. Generated associations may not transfer to real customers, and an intervention can change the behavior being predicted. A real policy also requires privacy and fairness review. Prediction alone does not identify what action helps.